# Market Model Estimation
Estimate alpha and beta per firm.

In [1]:
import sys
import pandas as pd
import numpy as np

sys.path.append('../src')
import event_study

# Load data
df_sec = pd.read_csv('../data/processed/sec_insider_trades.csv')
prices = pd.read_csv('../data/processed/stock_prices.csv', index_col='Date', parse_dates=True)
benchmark = pd.read_csv('../data/processed/market_benchmark.csv', index_col='Date', parse_dates=True)

# Calculate Returns
firm_returns = prices.pct_change().dropna()
market_col = '^GSPC' if '^GSPC' in benchmark.columns else benchmark.columns[0]
market_returns = benchmark[market_col].pct_change().dropna()

# Process events
df_sec['TRANS_DATE'] = pd.to_datetime(df_sec['TRANS_DATE'])
events = []

print("Estimating market models for events...")
for idx, row in df_sec.iterrows():
    ticker = row['ISSUERTRADINGSYMBOL']
    event_date = row['TRANS_DATE']
    
    if ticker not in firm_returns.columns:
        continue
        
    f_ret = firm_returns[ticker]
    
    alpha, beta = event_study.estimate_market_model(f_ret, market_returns, event_date)
    
    if alpha is not None and beta is not None:
        row_dict = row.to_dict()
        row_dict['alpha'] = alpha
        row_dict['beta'] = beta
        events.append(row_dict)

events_df = pd.DataFrame(events)
events_df.to_csv('../data/processed/events_with_market_model.csv', index=False)
print(f"Processed {len(events_df)} events with market models.")


C:\Users\ssr11\AppData\Local\Temp\ipykernel_8848\3370557598.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_sec['TRANS_DATE'] = pd.to_datetime(df_sec['TRANS_DATE'])


Estimating market models for events...
Processed 3037 events with market models.
